## Лабораторная работа № 1
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1. Очистка данных

В этой части работы мы:
* подключимся к кластеру `Apache Hadoop` и запустим `Spark`-сессию поверх `YARN`;
* загрузим датасет из распределённой файловой системы `HDFS`;
* отберем нужные признаки и приведём их к корректным типам;
* сохраним очищенные данные в таблицу `Apache Iceberg`.

### Подключение библиотек

* `SparkSession` — точка входа в Spark: через неё создаётся сессия и выполняются все операции;
* `SparkConf` — объект с настройками нашей сессии (режим работы, ресурсы, пакеты);
* `functions as F` — функции для работы со столбцами: приведение типов, даты, агрегаты.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql import functions as F

### Конфигурация Spark-сессии

Ключевые параметры и зачем они нужны:

| Параметр | Значение | Смысл |
|---|---|---|
| `spark.master` | `yarn` | Исполнители будут запущены не локально, а планировщиком `YARN` на узлах кластера |
| `spark.submit.deployMode` | `client` | Драйвер работает на нашей машине (в этой ноутбучной сессии), исполнители — на кластере |
| `spark.executor.*` | 2 × (12 ГБ, 8 ядер) | Ресурсы каждого исполнителя — процесса, который реально считает данные |
| `spark.driver.*` | 4 ГБ, 2 ядра | Ресурсы драйвера — он собирает результаты и держит контекст сессии |
| `spark.jars.packages` | `iceberg-spark-runtime-3.5` | Подтягивает поддержку табличного формата `Apache Iceberg` из Maven |
| `spark.sql.catalog.*` | `spark_catalog` | Настраивает каталог таблиц: тип `hadoop`, хранилище — `HDFS` в каталоге пользователя |

In [2]:
def create_spark_configuration() -> SparkConf:
    """Создаёт и настраивает конфигурацию Spark-сессии для работы на кластере."""
    # Имя пользователя — от него зависит путь к нашему хранилищу таблиц в HDFS
    user_name = os.getenv("USER")

    conf = SparkConf()
    conf.setAppName("SOBD lab1 — flights data cleansing")
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")

    # Ресурсы исполнителей и драйвера
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")

    # Подключаем Apache Iceberg — табличный формат для больших данных
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    return conf

In [3]:
conf = create_spark_configuration()

### Создание Spark-сессии

При создании сессии происходит самое интересное:
1. Драйвер связывается с `ResourceManager` кластера и просит ресурсы;
2. `YARN` выделяет контейнеры на узлах кластера и запускает в них **исполнителей** (executor);
3. Скачиваются пакеты Iceberg — первый запуск может занять пару минут.

Таблица ниже сессии (`spark`) — краткая информация о выделенных ресурсах.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

:: loading settings :: url = jar:file:/opt/spark-3.5.2-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/user0/.ivy2/cache
The jars for the packages stored in: /home/user0/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-09dec299-43c9-4c03-a80b-875c34b0118e;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 in central
:: resolution report :: resolve 563ms :: artifacts dl 21ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

### Загрузка датасета из HDFS

Используется датасет **«Reporting Carrier On-Time Performance», 2019–2023** — официальная статистика
Министерства транспорта США ([transtats.bts.gov](https://www.transtats.bts.gov/)) о пунктуальности
внутренних авиарейсов: около **32 млн рейсов** за 5 лет, ~14 ГБ в формате CSV.

Данные лежат в HDFS как директория из 60 файлов (по одному на месяц).
Указываем Spark **директорию целиком**: каждый файл будет прочитан отдельной задачей параллельно —
в этом и смысл распределённой обработки: данные не гоняются через один узел, а читаются там, где лежат.

> **Ленивые вычисления.** `spark.read` не читает данные сразу — Spark лишь строит план.
> Реальное чтение запустится при первом «действии» (action), например `show()` или `count()`.

In [5]:
path = "hdfs:///user/user0/sidskii_SOBD/data/csv"

df = (
    spark.read.format("csv")
    .option("header", "true")   # первая строка каждого файла — названия столбцов
    .load(path)
)

In [6]:
# Смотрим на первые строки. show() — «действие»: вот здесь данные реально прочитаются
df.show(5)

26/11/20 12:38:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+-------------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+-----------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------------+-----------------+-------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+------------+-------------+---------------+------------------+--------------+--------------------+-----------+-----------+-----------+-------------+----------------+------------+--------------+-----------

In [7]:
# Количество записей в датасете
df.count()

31682936

### Отбор признаков

В исходных данных **110 столбцов** — много служебных (коды FIPS, Wac, последовательные ID аэропортов,
детали диверсий с 70-го столбца). Для ясности и скорости оставим 27 информативных признаков:

| Группа | Столбцы | Смысл |
|---|---|---|
| Дата и маршрут | `FlightDate`, `DayOfWeek`, `Reporting_Airline`, `Origin`, `OriginCityName`, `Dest`, `DestCityName` | Когда, кто и куда летел (категориальные) |
| Плановое расписание | `CRSDepTime`, `CRSArrTime`, `CRSElapsedTime`, `Distance` | Запланированное время (формат ЧЧММ) и дальность |
| Фактический рейс | `DepTime`, `ArrTime`, `ActualElapsedTime`, `AirTime`, `TaxiIn`, `TaxiOut` | Что произошло на самом деле |
| **Задержки (наши цели)** | `DepDelay`, `ArrDelay` | Задержка вылета/прилёта в минутах |
| Статус рейса | `Cancelled`, `CancellationCode`, `Diverted` | Отменён ли, причина, посажен ли в другой аэропорт |
| Причины задержки | `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay` | Разбивка задержки по вине авиакомпании/погоды/УВД/безопасности/предыдущего рейса |

In [8]:
df = df.select(
    "FlightDate", "DayOfWeek", "Reporting_Airline",
    "Origin", "OriginCityName", "Dest", "DestCityName",
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime", "Distance",
    "DepTime", "ArrTime", "ActualElapsedTime", "AirTime", "TaxiIn", "TaxiOut",
    "DepDelay", "ArrDelay",
    "Cancelled", "CancellationCode", "Diverted",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
)
df.show(5)

+----------+---------+-----------------+------+-------------------+----+-----------------+----------+----------+--------------+--------+-------+-------+-----------------+-------+------+-------+--------+--------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|FlightDate|DayOfWeek|Reporting_Airline|Origin|     OriginCityName|Dest|     DestCityName|CRSDepTime|CRSArrTime|CRSElapsedTime|Distance|DepTime|ArrTime|ActualElapsedTime|AirTime|TaxiIn|TaxiOut|DepDelay|ArrDelay|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----------+---------+-----------------+------+-------------------+----+-----------------+----------+----------+--------------+--------+-------+-------+-----------------+-------+------+-------+--------+--------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|2019-01-04|        5|               OO|   SBP|San Luis

### Приведение типов данных

CSV — текстовый формат, поэтому Spark прочитал **все столбцы как строки**. Проверим схему
и приведём признаки к правильным типам: минуты и коды — `int`, дата — `date`.
Категориальные признаки (`Reporting_Airline`, `Origin`, `Dest`, города, код причины отмены)
оставим строками — их мы закодируем на этапе машинного обучения (лаба 2).

In [9]:
# До приведения: все столбцы — string
df.printSchema()

root
 |-- FlightDate: string (nullable = true)
 |-- DayOfWeek: string (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- CRSDepTime: string (nullable = true)
 |-- CRSArrTime: string (nullable = true)
 |-- CRSElapsedTime: string (nullable = true)
 |-- Distance: string (nullable = true)
 |-- DepTime: string (nullable = true)
 |-- ArrTime: string (nullable = true)
 |-- ActualElapsedTime: string (nullable = true)
 |-- AirTime: string (nullable = true)
 |-- TaxiIn: string (nullable = true)
 |-- TaxiOut: string (nullable = true)
 |-- DepDelay: string (nullable = true)
 |-- ArrDelay: string (nullable = true)
 |-- Cancelled: string (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: string (nullable = true)
 |-- CarrierDelay: string (nullable = true)
 |-- WeatherDelay: string (nullab

In [10]:
int_columns = [
    "DayOfWeek", "CRSDepTime", "CRSArrTime", "CRSElapsedTime", "Distance",
    "DepTime", "ArrTime", "ActualElapsedTime", "AirTime", "TaxiIn", "TaxiOut",
    "DepDelay", "ArrDelay", "Cancelled", "Diverted",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
]

# Цепочка withColumn: каждый вызов возвращает новый DataFrame с заменённым столбцом
df = df.withColumn("FlightDate", F.to_date("FlightDate", "yyyy-MM-dd"))
for column in int_columns:
    df = df.withColumn(column, F.col(column).cast("int"))

df.printSchema()

root
 |-- FlightDate: date (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- CRSElapsedTime: integer (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- ActualElapsedTime: integer (nullable = true)
 |-- AirTime: integer (nullable = true)
 |-- TaxiIn: integer (nullable = true)
 |-- TaxiOut: integer (nullable = true)
 |-- DepDelay: integer (nullable = true)
 |-- ArrDelay: integer (nullable = true)
 |-- Cancelled: integer (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: integer (nullable = true)
 |-- CarrierDelay: integer (nullable = true)
 |-- WeatherDelay: 

In [11]:
df.show(5)

+----------+---------+-----------------+------+-------------------+----+-----------------+----------+----------+--------------+--------+-------+-------+-----------------+-------+------+-------+--------+--------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|FlightDate|DayOfWeek|Reporting_Airline|Origin|     OriginCityName|Dest|     DestCityName|CRSDepTime|CRSArrTime|CRSElapsedTime|Distance|DepTime|ArrTime|ActualElapsedTime|AirTime|TaxiIn|TaxiOut|DepDelay|ArrDelay|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----------+---------+-----------------+------+-------------------+----+-----------------+----------+----------+--------------+--------+-------+-------+-----------------+-------+------+-------+--------+--------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|2019-01-04|        5|               OO|   SBP|San Luis

### Сохранение в таблицу Apache Iceberg

`Apache Iceberg` — открытый табличный формат для озёр данных: поверх файлов в HDFS он добавляет
схему, транзакции (ACID), снапшоты версий и эффективные запросы по метаданным.
В отличие от «голых» CSV, таблицу можно обновлять и читать без пересчёта всего.

Схема работы: создадим свою базу данных в каталоге Spark и запишем туда таблицу `lab1_flights_table`.

In [12]:
database_name = "sidskii_database"

spark.sql(f"CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}")
spark.catalog.setCurrentDatabase(database_name)

In [13]:
# Запись таблицы (create — создать новую; при повторном запуске таблицу надо сначала удалить)
df.writeTo("lab1_flights_table").using("iceberg").create()

# Если таблица уже существует из прошлого запуска, раскомментируйте:
# spark.sql("DROP TABLE IF EXISTS spark_catalog.sidskii_database.lab1_flights_table")

In [14]:
for table in spark.catalog.listTables():
    print(table.name)

lab1_flights_table


### Завершение работы

Останавливаем сессию — YARN освобождает контейнеры с исполнителями.
Таблица `lab1_flights_table` осталась в HDFS и будет использована во второй части лабораторной работы.

In [15]:
spark.stop()